# M3 training runs

We train one or two run configs from `configs/m3/` at a time, each as its own `python -m serpentguard.train` process
on its own GPU. Every M3 training experiment runs through this notebook: E1 to E4, the N0 noise floor, the FE front
ends, K1, the H1 retrains and N1. Each config's run id names the question it answers.

**Inputs:** `serpentguard-indian-snakes-clean`, `serpentguard-sam-masks` and `serpentguard-m3-subset`, found with
`kagglehub.dataset_download`. Front-end runs also read notebook 13's predictions: from the dataset
`serpentguard-m3-frontend-preds`, from the notebook 13 kernel's output (`PREDS_KERNEL`, declared as a kernel source)
or from a private dataset copy of it (`PREDS_DATASET`); without any of them `PREDS` is None. The code comes from
`../src` and `../configs/m3`, which the runner kernel unpacks into `/kaggle/working/repo`.

**Outputs:** each run writes to `/kaggle/working/<run_id>/` and streams its output to `train.log` there.
`/kaggle/working/summary.csv` holds the runs' headline numbers.

**Run:** on Kaggle with one GPU per config. `RUN_CONFIGS` names one or two configs, and `EPOCHS` overrides their
epochs. We pin albumentations, timm and scikit-learn to `requirements-m3.txt` and read every input file once into the
page cache before the runs start.

In [1]:
RUN_CONFIGS = ['H1_top1_s1', 'H1_top1_s2']
EPOCHS = None                          # None keeps each config's epochs
PREDS_KERNEL = 'puneetbajajiitm/sg-m3-frontends'
PREDS_DATASET = None  # "owner/slug" of a private dataset holding a copy of frontend_preds.csv, for another account


In [2]:
import os
import subprocess
import sys

ALBUMENTATIONS_VERSION = "2.0.8"  # requirements-m3.txt; the per-transform augmentation seeding was verified on it
TIMM_VERSION = "1.0.31"            # requirements-m3.txt; the encoders' feature maps were measured on it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"albumentations=={ALBUMENTATIONS_VERSION}",
                f"timm=={TIMM_VERSION}"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # otherwise albumentations asks PyPI for its latest version on import

import albumentations
import timm

assert albumentations.__version__ == ALBUMENTATIONS_VERSION, f"albumentations {albumentations.__version__} is loaded"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} is loaded"
print(f"albumentations {albumentations.__version__}, timm {timm.__version__}")

albumentations 2.0.8, timm 1.0.31


In [3]:
SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt; the runner's metrics use it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, (f"scikit-learn {sklearn.__version__} is loaded; restart the "
                                                     f"kernel after the install")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.1


In [4]:
import json
import time
from pathlib import Path

import kagglehub

CODE = Path("/kaggle/input/serpentguard-code/src")
CODE = CODE if CODE.exists() else Path("../src").resolve()
sys.path.insert(0, str(CODE))
CONFIGS = CODE.parent / "configs" / "m3"

import pandas as pd
import torch

from serpentguard.config import load_config
from serpentguard.models import MOBILE_ENCODERS

OWNER = "puneetbajajiitm"


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


DATA = input_root("serpentguard-indian-snakes-clean")
MASKS = input_root("serpentguard-sam-masks")
SUBSET = input_root("serpentguard-m3-subset") / "subset.csv"
if PREDS_KERNEL:  # a batch kernel reads another notebook's output only when it is declared as a kernel source
    PREDS = next(Path(kagglehub.notebook_output_download(PREDS_KERNEL)).rglob("frontend_preds.csv")).parent
elif PREDS_DATASET:  # a byte-identical copy owned by this account, since another account's kernel output is private
    PREDS = next(Path(kagglehub.dataset_download(PREDS_DATASET)).rglob("frontend_preds.csv")).parent
else:
    try:
        PREDS = input_root("serpentguard-m3-frontend-preds")  # only front-end runs read it
    except Exception as error:  # neither attached nor downloadable
        print(f"serpentguard-m3-frontend-preds unavailable ({type(error).__name__}: {error})")
        PREDS = None
OUT = Path("/kaggle/working")
POLL_SECONDS = 60
EXPECTED_SUBSET_HASH = "40196d19a68e"  # first 12 hex digits of the real subset's SHA-256

for required in (DATA / "metadata.csv", DATA / "classes.csv", DATA / "data", MASKS / "mask_stats.csv", SUBSET):
    assert required.exists(), f"missing input {required}"
missing_encoders = sorted(set(MOBILE_ENCODERS) - set(timm.list_models()))
assert not missing_encoders, f"timm {timm.__version__} has no {missing_encoders}"
gpus = torch.cuda.device_count()
assert 1 <= len(RUN_CONFIGS) <= gpus, f"{len(RUN_CONFIGS)} configs need as many GPUs; this session has {gpus}"
configs = {name: load_config(CONFIGS / f"{name}.yaml") for name in RUN_CONFIGS}
workers = max(1, (os.cpu_count() or 2) // len(RUN_CONFIGS))
print(f"code {CODE}; {gpus} GPUs, {os.cpu_count()} CPUs, {workers} loader workers per run; "
      f"front-end predictions {PREDS or 'not attached'}")

code /kaggle/working/repo/src; 2 GPUs, 4 CPUs, 2 loader workers per run; front-end predictions /kaggle/input/notebooks/puneetbajajiitm/sg-m3-frontends/frontend_preds


In [5]:
from serpentguard.run_data import WARM_THREADS, run_files, warm_page_cache

# A cold read on the dataset mount takes about 60 ms. Reading every file the runs open once, 64 at a time, puts them
# in the page cache that every process shares, so the runs' loaders read from memory.
files = sorted({path for cfg in configs.values() for path in run_files(cfg, DATA, MASKS, SUBSET, PREDS)})
warmed = warm_page_cache(files, threads=WARM_THREADS)
print(f"page cache: {warmed['files']:,} files ({warmed['missing']} missing), {warmed['mb']:,.0f} MB in "
      f"{warmed['seconds']:.0f} s, {warmed['mb'] / max(warmed['seconds'], 1e-9):.0f} MB/s")

page cache: 39,212 files (0 missing), 5,386 MB in 186 s, 29 MB/s


In [6]:
runs = {}
for gpu, (name, cfg) in enumerate(configs.items()):
    out = OUT / cfg.run_id
    out.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, "-u", "-m", "serpentguard.train", "--config", str(CONFIGS / f"{name}.yaml"),
               "--data-root", str(DATA), "--masks-root", str(MASKS), "--subset", str(SUBSET), "--out", str(out),
               "--device", "cuda:0", "--num-workers", str(workers)]
    if PREDS is not None:
        command += ["--preds-root", str(PREDS)]
    if EPOCHS is not None:
        command += ["--epochs", str(EPOCHS)]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu),
           "PYTHONPATH": os.pathsep.join(filter(None, [str(CODE), os.environ.get("PYTHONPATH")]))}
    log = open(out / "train.log", "w")
    runs[cfg.run_id] = (subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=env), log)
    print(f"{cfg.run_id} on GPU {gpu}: {' '.join(command)}")

H1_top1_s1 on GPU 0: /usr/bin/python3 -u -m serpentguard.train --config /kaggle/working/repo/configs/m3/H1_top1_s1.yaml --data-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-indian-snakes-clean --masks-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-sam-masks --subset /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-subset/subset.csv --out /kaggle/working/H1_top1_s1 --device cuda:0 --num-workers 2 --preds-root /kaggle/input/notebooks/puneetbajajiitm/sg-m3-frontends/frontend_preds
H1_top1_s2 on GPU 1: /usr/bin/python3 -u -m serpentguard.train --config /kaggle/working/repo/configs/m3/H1_top1_s2.yaml --data-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-indian-snakes-clean --masks-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-sam-masks --subset /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-subset/subset.csv --out /kaggle/working/H1_top1_s2 --device cuda:0 --num-workers 2 --preds-root /kaggle/input/notebooks/puneetbajajiitm/sg-m3-front

In [7]:
def last_line(path):
    lines = path.read_text(errors="replace").strip().splitlines()
    return lines[-1] if lines else "(no output yet)"


started = time.time()
while any(process.poll() is None for process, _ in runs.values()):
    time.sleep(POLL_SECONDS)
    for run_id in runs:
        print(f"{(time.time() - started) / 60:5.1f} min  {run_id}: {last_line(OUT / run_id / 'train.log')}")
for run_id, (process, log) in runs.items():
    log.close()
    print(f"{run_id} exited with {process.returncode}")
    if process.returncode:
        print((OUT / run_id / "train.log").read_text(errors="replace")[-5000:])

  1.0 min  H1_top1_s1:   scheduler.step()
  1.0 min  H1_top1_s2:   scheduler.step()
  2.0 min  H1_top1_s1:   scheduler.step()
  2.0 min  H1_top1_s2:   scheduler.step()
  3.0 min  H1_top1_s1:   scheduler.step()
  3.0 min  H1_top1_s2:   scheduler.step()
  4.0 min  H1_top1_s1:   scheduler.step()
  4.0 min  H1_top1_s2:   scheduler.step()
  5.0 min  H1_top1_s1:   scheduler.step()
  5.0 min  H1_top1_s2:   scheduler.step()
  6.0 min  H1_top1_s1:   scheduler.step()
  6.0 min  H1_top1_s2:   scheduler.step()
  7.0 min  H1_top1_s1:   scheduler.step()
  7.0 min  H1_top1_s2:   scheduler.step()
  8.0 min  H1_top1_s1:   scheduler.step()
  8.0 min  H1_top1_s2:   scheduler.step()
  9.0 min  H1_top1_s1:   scheduler.step()
  9.0 min  H1_top1_s2:   scheduler.step()
 10.0 min  H1_top1_s1: H1_top1_s1 epoch 2/20: macro-F1 0.209, top-1 0.483, top-3 0.653, loss 13.619, lr 9.47e-04, 73 photos/s, data wait 14%, epoch 186 s + eval 133 s
 10.0 min  H1_top1_s2: H1_top1_s2 epoch 2/20: macro-F1 0.203, top-1 0.485, to

In [8]:
rows = []
for run_id in runs:
    final_path = OUT / run_id / "final_metrics.json"
    if not final_path.exists():
        continue
    final = json.loads(final_path.read_text())
    run, head, safety, timing = final["run"], final["head"], final["safety"], final["timing"]
    assert run["subset_hash"].startswith(EXPECTED_SUBSET_HASH), f"{run_id} read another subset: {run['subset_hash']}"
    rows.append({"run": run_id, "best epoch": run["best_epoch"], "train photos": run["train_photos"],
                 "macro-F1 head": head["macro_f1"], "macro-F1 prototype": final["prototype"]["macro_f1"],
                 "macro-F1 balanced": final["prototype_balanced"]["macro_f1"], "top-1": head["top1"],
                 "top-3": head["top3"], "WHO recall (O2 raw)": safety["who_recall_raw"],
                 "listed as not listed (O3 raw)": safety["top1_error_raw"], "ECE": final["calibration"]["ece"],
                 "min/epoch": timing["seconds_per_epoch"] / 60, "minutes": timing["total_seconds"] / 60,
                 "photos/s": timing["photos_per_second"], "data wait": timing["data_wait_share"],
                 "peak GPU MB": timing["peak_gpu_memory_mb"],
                 "val prediction coverage": (run["prediction_coverage"] or {}).get("val")})
summary = pd.DataFrame(rows).set_index("run") if rows else pd.DataFrame()
summary.to_csv(OUT / "summary.csv")
failed = [run_id for run_id, (process, _) in runs.items() if process.returncode]
assert not failed, f"runs failed: {failed}; their log tails are above"
summary.round(3)

,best epoch,train photos,macro-F1 head,macro-F1 prototype,macro-F1 balanced,top-1,top-3,WHO recall (O2 raw),listed as not listed (O3 raw),ECE,min/epoch,minutes,photos/s,data wait,peak GPU MB,val prediction coverage
run,,,,,,,,,,,,,,,,
H1_top1_s1,18,13611,0.367,0.356,0.356,0.622,0.762,0.768,0.232,0.211,3.089,86.528,73.212,0.134,10733.396,1.0
H1_top1_s2,18,13611,0.367,0.362,0.363,0.618,0.763,0.763,0.237,0.214,3.063,86.203,73.831,0.169,10733.396,1.0
